# EDA 03: Events and announcements

Do company events move prices, and when can the system know about them?

- **Prices:** as in E02: raw closes with each stock's returns dropped on every corporate-action ex-date
  (adjusted prices, task 2.4, don't exist yet). Abnormal return = stock log return − Nifty 50 log return.
- **Event day (day 0):** the first trading day whose 08:30 IST forecast may use the record: published before
  08:30 IST on a trading day → that day, otherwise the next trading day. Trading days are the price dates.
- **Period:** counts over the full range; event studies 2016–2023 (windows stop at 29 Dec 2023, so nothing
  reaches the 2024 test period).
- **Results** come from `financial_results` (exact broadcast time; standalone and consolidated filings merged
  into one event per stock and quarter, at the earliest broadcast). "Financial Result Updates" announcements
  stay in the category analysis as a cross-check.
- **Intervals:** 95% bootstrap intervals that resample whole calendar weeks, because events in the same
  results season move together. Each event set has its own random-day control: the same number of days per
  stock, drawn from days with a complete window (fixed seed).
- **Helpers:** `src/analysis/event_study.py` (reused by task 9.1).

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src.analysis import event_study as E
from src.analysis import inventory as inv
from src.analysis import returns as R
from src.analysis.plots import heatmap, save
from src.config import Config

pd.set_option("display.width", 220, "display.max_columns", 30, "display.max_colwidth", 60, "display.precision", 4)
cfg = Config.load()
START, END = "2016-01-01", "2023-12-31"
PRE, POST, NBOOT, SEED = 5, 10, 2000, 7
con = duckdb.connect(str(cfg.duckdb_path), read_only=True)
con.execute("SET TimeZone = 'UTC'")

prices = con.execute("SELECT symbol, trade_date, open, close FROM prices_daily WHERE trade_date BETWEEN ? AND ?", [START, END]).df()
ca = con.execute("SELECT symbol, ex_date, ca_type FROM corporate_actions WHERE ex_date BETWEEN ? AND ?", [START, END]).df()
nifty = con.execute("SELECT trade_date, open, close FROM index_daily WHERE index_name = 'Nifty 50' AND trade_date BETWEEN ? AND ?",
                    [START, END]).df().assign(symbol="NIFTY")
ann_all = con.execute("SELECT doc_id, symbol, category, published_at, is_important FROM announcements").df()
fr = con.execute("SELECT symbol, period_to, consolidated, broadcast_at FROM financial_results").df()
ins = con.execute("SELECT symbol, person_category, acq_mode, buy_value, sell_value, quantity, published_at FROM insider_trades").df()
sh = inv.load_table(con, inv.TABLES["shareholding"])

C, O = R.to_wide(prices, "close"), R.to_wide(prices, "open")
MASK = R.ex_date_mask(C.index, C.columns, ca)
r = R.log_returns(C, MASK)
NC, NO = R.to_wide(nifty, "close"), R.to_wide(nifty, "open")
rn = R.log_returns(NC)["NIFTY"]
ar = E.abnormal_returns(r, rn)
gap, intra = R.gap_intraday(O, C, MASK)
ngap, nintra = R.gap_intraday(NO, NC)
ar_gap, ar_intra = gap.sub(ngap["NIFTY"], axis=0), intra.sub(nintra["NIFTY"], axis=0)
DAYS = C.index
ann_all["category"] = ann_all["category"].str.strip()
print(f"{C.shape[1]} stocks, {len(DAYS)} trading days; announcements (all years): {len(ann_all):,}; "
      f"results rows: {len(fr):,}; insider rows: {len(ins):,}")

In [ ]:
# Event-study machinery used by every section below
def study(events, seed):
    """Complete windows for the events and for a matched random-day control."""
    ev = events.dropna(subset=["event_day"]).reset_index(drop=True)
    w = E.complete(E.event_windows(ar, ev, PRE, POST))
    ev = ev.loc[w.index]
    ctl = E.control_sample(ar, ev, PRE, POST, seed=seed)
    wc = E.complete(E.event_windows(ar, ctl, PRE, POST))
    return {"ev": ev, "w": w, "ctl": ctl.loc[wc.index], "wc": wc}

def measures(w):
    c01 = E.car(w, 0, 1)
    return pd.DataFrame({"CAR[-5,-1]": E.car(w, -5, -1), "AR[0]": w[0], "CAR[0,+1]": c01,
                         "CAR[0,+5]": E.car(w, 0, 5), "CAR[0,+10]": E.car(w, 0, 10), "|CAR[0,+1]|": c01.abs()})

def compare(st, seed=SEED):
    """Per measure: event mean with interval, share positive, control mean, and whether the difference excludes 0."""
    m, mc = measures(st["w"]), measures(st["wc"])
    cl, clc = E.week_cluster(st["ev"]["event_day"]), E.week_cluster(st["ctl"]["event_day"])
    s, sc = E.summarize(m, cl, NBOOT, seed), E.summarize(mc, clc, NBOOT, seed)
    d = E.difference(m, cl, mc, clc, NBOOT, seed)
    return pd.DataFrame({"events": s["n"], "mean": s["mean"], "lo": s["lo"], "hi": s["hi"], "share_pos": s["share_pos"],
                         "control mean": sc["mean"], "control share_pos": sc["share_pos"],
                         "diff lo": d["lo"], "diff hi": d["hi"], "beyond control": (d["lo"] > 0) | (d["hi"] < 0)})

def path(st, which="w", ev="ev"):
    p = E.car_path(st[which])
    return E.summarize(p, E.week_cluster(st[ev]["event_day"]), NBOOT, SEED)

def plot_path(ax, st, title):
    for which, ev, color, lbl in [("w", "ev", "#4C72B0", "events"), ("wc", "ctl", "grey", "random-day control")]:
        p = path(st, which, ev)
        ax.plot(p.index, p["mean"] * 100, color=color, marker="o", ms=2.5, label=f"{lbl} (n={len(st[which]):,})")
        ax.fill_between(p.index, p["lo"] * 100, p["hi"] * 100, color=color, alpha=0.2)
    ax.axvline(-0.5, color="crimson", ls="--", lw=0.8); ax.axhline(0, color="k", lw=0.6)
    ax.set_title(title, fontsize=9); ax.set_xlabel("trading days from day 0"); ax.set_ylabel("mean CAR (%)")
    ax.legend(fontsize=7)

FMT = {"events": "{:,.0f}", "mean": "{:+.2%}", "lo": "{:+.2%}", "hi": "{:+.2%}", "share_pos": "{:.1%}",
       "control mean": "{:+.2%}", "control share_pos": "{:.1%}", "diff lo": "{:+.2%}", "diff hi": "{:+.2%}"}

## 1. Volume: announcements per year, stock and category

In [ ]:
ann_all["year"] = ann_all.published_at.dt.tz_convert(inv.IST).dt.year
per_year = ann_all.groupby("year").agg(announcements=("doc_id", "size"), important=("is_important", "sum"))
per_year["important share"] = per_year.important / per_year.announcements
display(per_year.T.style.format("{:,.2f}"))

ann = ann_all[(ann_all.published_at >= pd.Timestamp(START, tz="UTC")) & (ann_all.published_at < pd.Timestamp("2024-01-01", tz="UTC"))].copy()
print(f"2016-2023: {len(ann):,} announcements, {ann.is_important.mean():.1%} important, "
      f"{ann.category.nunique()} categories; per stock median {ann.groupby('symbol').size().median():.0f} "
      f"(min {ann.groupby('symbol').size().min()}, max {ann.groupby('symbol').size().max()})")
cats = ann.groupby("category").agg(n=("doc_id", "size"), important=("is_important", "mean")).sort_values("n", ascending=False)
print("Most common categories (2016-2023):")
display(cats.head(15).style.format({"n": "{:,.0f}", "important": "{:.0%}"}))
imp_cats = ann[ann.is_important].category.value_counts()
print("Most common important categories:")
display(imp_cats.head(15).to_frame("n"))

fig, axes = plt.subplots(1, 3, figsize=(18, 5), gridspec_kw={"width_ratios": [1, 1.4, 1.4]})
per_year[["important"]].assign(routine=per_year.announcements - per_year.important).plot.bar(
    stacked=True, ax=axes[0], color=["crimson", "#BBBBBB"], width=0.8)
axes[0].set_title("Announcements per year", fontsize=10); axes[0].set_xlabel("")
imp_cats.head(15)[::-1].plot.barh(ax=axes[1], color="crimson"); axes[1].set_title("Top important categories, 2016-2023", fontsize=10)
axes[1].tick_params(axis="y", labelsize=7)
ann.groupby("symbol").size().sort_values().plot.bar(ax=axes[2], color="#4C72B0", width=0.8)
axes[2].set_title("Announcements per stock, 2016-2023", fontsize=10); axes[2].tick_params(axis="x", labelsize=6)
fig.tight_layout(); save(fig, "03_volume"); plt.show()

## 2. Event day: which trading day can each record first inform?

In [ ]:
ann["event_day"] = E.event_days(ann.published_at, DAYS)
pub_ist = ann.published_at.dt.tz_convert(inv.IST)
ann["session"] = inv.ist_session(ann.published_at)
ann["same_day"] = ann.event_day.dt.date.values == pub_ist.dt.date.values
ann["lag_days"] = (ann.event_day - pub_ist.dt.tz_localize(None).dt.normalize()).dt.days
print(f"announcements with an event day: {ann.event_day.notna().mean():.2%} (the rest fall after the last price day)")
print(f"inform the same day: {ann.same_day.mean():.1%}; next calendar day: {(ann.lag_days == 1).mean():.1%}; "
      f"2+ calendar days later (weekends, holidays): {(ann.lag_days >= 2).mean():.1%}")
top = imp_cats.head(10).index
tab = ann[ann.category.isin(top)].groupby("category").agg(
    n=("doc_id", "size"), before_0830=("session", lambda s: (s == "before 08:30").mean()),
    market_hours=("session", lambda s: (s == "08:30-15:30").mean()), after_1530=("session", lambda s: (s == "after 15:30").mean()),
    same_day=("same_day", "mean"), median_lag_days=("lag_days", "median")).sort_values("n", ascending=False)
display(tab.style.format({"n": "{:,.0f}", "before_0830": "{:.1%}", "market_hours": "{:.1%}", "after_1530": "{:.1%}",
                          "same_day": "{:.1%}", "median_lag_days": "{:.0f}"}))

## 3. Event study: cumulative abnormal returns from day −5 to +10

(a) results filings, (b) important announcements by category (categories with ≥ 50 events, one per stock,
day and category), (c) a matched random-day control for each set. `beyond control` = the 95% interval of
(event mean − control mean) excludes 0.

In [ ]:
res = (fr.dropna(subset=["broadcast_at"]).groupby(["symbol", "period_to"], as_index=False).broadcast_at.min())
res["event_day"] = E.event_days(res.broadcast_at, DAYS)
res = res[res.event_day >= pd.Timestamp(START)]
studies = {"Results (financial_results)": study(res, SEED)}

imp = ann[ann.is_important].drop_duplicates(["symbol", "event_day", "category"])
studies["All important announcements"] = study(imp, SEED + 1)
for i, (cat, n) in enumerate(imp.category.value_counts().items()):
    st = study(imp[imp.category == cat], SEED + 10 + i)
    if len(st["w"]) >= 50:
        studies[cat] = st
res_all = res.dropna(subset=["event_day"]).reset_index(drop=True)
w_all = E.event_windows(ar, res_all, PRE, POST)
hits_ex = E.event_windows(MASK.astype(float), res_all, PRE, POST).fillna(0).sum(axis=1) > 0
edge = E.event_windows(pd.DataFrame(1.0, index=DAYS, columns=C.columns), res_all, PRE, POST).isna().any(axis=1)
print(f"results: {len(res_all):,} stock-quarters with a day 0 in 2016-2023; full window: {len(studies['Results (financial_results)']['w']):,}; "
      f"lost because the window contains a dropped ex-date: {int((hits_ex & ~edge).sum())}; "
      f"because it runs past the data or the listing: {int(edge.sum())}")
print("event sets:", {k: len(v["w"]) for k, v in studies.items()})

comp = {k: compare(st) for k, st in studies.items()}
rows = []
for k, c in comp.items():
    row = {"set": k, "events": int(c.loc["CAR[0,+1]", "events"])}
    for mname in ["CAR[-5,-1]", "CAR[0,+1]", "CAR[0,+5]", "CAR[0,+10]", "|CAR[0,+1]|"]:
        row[mname] = f"{c.loc[mname, 'mean']:+.2%} [{c.loc[mname, 'lo']:+.2%}, {c.loc[mname, 'hi']:+.2%}]" + (" *" if c.loc[mname, "beyond control"] else "")
    row["share CAR[0,+5] > 0"] = c.loc["CAR[0,+5]", "share_pos"]
    row["control |CAR[0,+1]|"] = c.loc["|CAR[0,+1]|", "control mean"]
    rows.append(row)
overview = pd.DataFrame(rows).set_index("set")
print("Mean [95% interval]; * = beyond the random-day control")
display(overview.style.format({"share CAR[0,+5] > 0": "{:.1%}", "control |CAR[0,+1]|": "{:.2%}"}))

In [ ]:
show = list(studies)[:8]
fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharex=True)
for ax, k in zip(axes.flat, show):
    plot_path(ax, studies[k], k)
fig.tight_layout(); save(fig, "03_event_study"); plt.show()

In [ ]:
# Day 0 split: how much of the day-0 abnormal move happens overnight, before a 09:15 entry?
def day0_split(st, which="w", ev="ev"):
    e = st[ev]
    g = E.event_windows(ar_gap, e, 0, 0)[0]; i_ = E.event_windows(ar_intra, e, 0, 0)[0]
    ok = g.notna() & i_.notna()
    g, i_ = g[ok], i_[ok]
    return {"n": len(g), "mean |gap AR|": g.abs().mean(), "mean |intraday AR|": i_.abs().mean(),
            "gap share of variance": g.var() / (g.var() + i_.var()), "corr(gap, intraday)": g.corr(i_),
            "mean gap AR": g.mean(), "mean intraday AR": i_.mean()}

split = pd.DataFrame({f"{k} | {lbl}": day0_split(studies[k], w, e)
                      for k in ["Results (financial_results)", "All important announcements"]
                      for w, e, lbl in [("w", "ev", "events"), ("wc", "ctl", "control")]}).T
display(split.style.format({"n": "{:,.0f}", "mean |gap AR|": "{:.2%}", "mean |intraday AR|": "{:.2%}",
                            "gap share of variance": "{:.0%}", "corr(gap, intraday)": "{:+.2f}",
                            "mean gap AR": "{:+.3%}", "mean intraday AR": "{:+.3%}"}))

## 4. Results: release time and size of the reaction

A result released during market hours (08:30–15:30 IST) is day 0 the *next* trading day under the time rule,
so part of the reaction happens on day −1, before the system may use it.

In [ ]:
st_res = studies["Results (financial_results)"]
rev = st_res["ev"].copy()
rev["session"] = inv.ist_session(rev.broadcast_at)
rev["hour"] = rev.broadcast_at.dt.tz_convert(inv.IST).dt.hour + rev.broadcast_at.dt.tz_convert(inv.IST).dt.minute / 60
print("release session (2016-2023 results with full windows):", rev.session.value_counts(normalize=True).round(3).to_dict())
by_year_sess = pd.crosstab(rev.event_day.dt.year, rev.session, normalize="index")
display(by_year_sess.style.format("{:.0%}"))

absw = st_res["w"].abs()
ctl_abs = st_res["wc"].abs().mean()
react = pd.DataFrame({sess: absw.loc[rev.index[rev.session == sess]].mean() for sess in inv.SESSIONS}).loc[-2:3]
react["control"] = ctl_abs.loc[-2:3]
react.index.name = "offset"
print("Mean |AR| by offset, by release session (control = random days):")
display(react.style.format("{:.2%}"))
c = comp["Results (financial_results)"].loc["|CAR[0,+1]|"]
print(f"|CAR[0,+1]| results {c['mean']:.2%} [{c['lo']:.2%}, {c['hi']:.2%}] vs control {c['control mean']:.2%}: "
      f"{c['mean'] / c['control mean']:.2f}x")
abs01 = E.car(st_res["w"], -1, 1).abs()
print(f"|CAR[-1,+1]| (including the publication day) results {abs01.mean():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].hist(rev.hour, bins=48, range=(0, 24), color="#4C72B0")
for x in (8.5, 15.5):
    axes[0].axvline(x, color="crimson", ls="--", lw=1)
axes[0].set_title(f"Results release time, IST (n={len(rev):,}, 2016-2023)", fontsize=10); axes[0].set_xlabel("hour")
for sess in inv.SESSIONS:
    if (rev.session == sess).sum() >= 20:
        axes[1].plot(react.index, react[sess] * 100, marker="o", label=f"released {sess} (n={(rev.session == sess).sum()})")
axes[1].plot(react.index, react["control"] * 100, "k--", label="random days")
axes[1].set_title("Mean |abnormal return| around results, by release time", fontsize=10)
axes[1].set_xlabel("trading days from day 0"); axes[1].set_ylabel("%"); axes[1].legend(fontsize=8)
fig.tight_layout(); save(fig, "03_results_reaction"); plt.show()

In [ ]:
# Cross-check: the same results seen through "Financial Result Updates" announcements.
# For each results event, the nearest FRU announcement of the same stock within 10 trading days.
pos = {d: i for i, d in enumerate(DAYS)}
fru = ann[ann.category.isin(["Financial Result Updates", "Financial Results Updates"])].dropna(subset=["event_day"])
fru_pos = fru.groupby("symbol").event_day.apply(lambda s: np.array(sorted(pos[d] for d in s)))
rows = []
for sym, d, b in zip(rev.symbol, rev.event_day, rev.broadcast_at):
    arr = fru_pos.get(sym)
    if arr is None or len(arr) == 0:
        rows.append(np.nan); continue
    k = arr[np.abs(arr - pos[d]).argmin()] - pos[d]
    rows.append(k if abs(k) <= 10 else np.nan)
off = pd.Series(rows, index=rev.index)
print(f"results events with an FRU announcement within 10 trading days: {off.notna().mean():.1%}")
print("FRU day 0 minus results day 0 (trading days):", off.value_counts(normalize=True).sort_index().round(3).to_dict())
fp = fru[["symbol", "published_at"]].rename(columns={"published_at": "fru_at"}).sort_values("fru_at")
m = pd.merge_asof(rev.sort_values("broadcast_at"), fp, left_on="broadcast_at", right_on="fru_at", by="symbol", direction="nearest")
dt = (m.fru_at - m.broadcast_at).dt.total_seconds() / 3600
print(f"FRU published minus results broadcast (hours): median {dt.median():.2f}, share within 1 hour {(dt.abs() <= 1).mean():.1%}")
grp = pd.cut(off, [-11, -2, -1, 0], labels=["FRU 2-10 days earlier", "FRU 1 day earlier", "FRU same day 0"])
tab = st_res["w"].abs().groupby(grp, observed=True).mean()[[-2, -1, 0, 1]]
tab.insert(0, "events", grp.value_counts())
print("Mean |AR| around the financial_results day 0, by where the FRU announcement falls:")
display(tab.style.format({c: "{:.2%}" for c in [-2, -1, 0, 1]}))

## 5. Volatility: absolute return on event days vs other days

Raw daily stock returns (not abnormal) on day 0, vs the same stocks on random days.

In [ ]:
rows = []
for k, st in studies.items():
    e0 = E.event_windows(r, st["ev"], 0, 0)[0].abs()
    c0 = E.event_windows(r, st["ctl"], 0, 0)[0].abs()
    b = E.difference(e0.to_frame("x"), E.week_cluster(st["ev"].event_day), c0.to_frame("x"),
                     E.week_cluster(st["ctl"].event_day), NBOOT, SEED).loc["x"]
    rows.append({"set": k, "events": e0.notna().sum(), "mean |r| day 0": e0.mean(), "control mean |r|": c0.mean(),
                 "ratio": e0.mean() / c0.mean(), "diff lo": b["lo"], "diff hi": b["hi"], "beyond control": b["lo"] > 0 or b["hi"] < 0})
vol = pd.DataFrame(rows).set_index("set").sort_values("ratio", ascending=False)
display(vol.style.format({"events": "{:,.0f}", "mean |r| day 0": "{:.2%}", "control mean |r|": "{:.2%}", "ratio": "{:.2f}",
                          "diff lo": "{:+.2%}", "diff hi": "{:+.2%}"}))

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["crimson" if b else "#BBBBBB" for b in vol["beyond control"]]
ax.barh(range(len(vol)), vol.ratio, color=colors)
ax.set_yticks(range(len(vol)), [f"{i} (n={n:,})" for i, n in zip(vol.index, vol.events)], fontsize=8)
ax.invert_yaxis(); ax.axvline(1, color="k", lw=0.8)
ax.set_title("Mean |daily return| on day 0 / on random days (red: interval excludes no difference)", fontsize=10)
fig.tight_layout(); save(fig, "03_event_volatility"); plt.show()

## 6. Insider trades and shareholding

Coverage first, then (2016–2021, where insider data is dense) whether promoter market purchases or sales come
before abnormal returns. Day 0 is the first trading day after the **disclosure** (`published_at`), not the
trade date, which the market didn't know yet.

In [ ]:
ins["year"] = ins.published_at.dt.tz_convert(inv.IST).dt.year
promoter = ins.person_category.isin(["Promoters", "Promoter Group"])
market = ins.acq_mode.isin(["Market Purchase", "Market Sale"])
cov = ins.groupby("year").agg(rows=("symbol", "size"), stocks=("symbol", "nunique"))
cov["promoter market trades"] = ins[promoter & market].groupby("year").size()
cov["shareholding stocks"] = sh.groupby(sh.period_end.dt.year).symbol.nunique()
display(cov.fillna(0).astype(int).T)

pt = ins[promoter & market & (ins.published_at >= pd.Timestamp(START, tz="UTC")) & (ins.published_at < pd.Timestamp("2022-01-01", tz="UTC"))].copy()
pt["side"] = np.where(pt.acq_mode == "Market Purchase", "promoter buy", "promoter sell")
pt["event_day"] = E.event_days(pt.published_at, DAYS)
pt_ev = pt.drop_duplicates(["symbol", "event_day", "side"])
print("promoter market trades 2016-2021:", pt.side.value_counts().to_dict(), "| distinct stock-days:", pt_ev.side.value_counts().to_dict(),
      "| stocks:", pt_ev.groupby("side").symbol.nunique().to_dict())
ins_studies = {s: study(pt_ev[pt_ev.side == s], SEED + 100 + i) for i, s in enumerate(["promoter buy", "promoter sell"])}
for s, st in ins_studies.items():
    print(f"\n{s}: {len(st['w'])} events with full windows, from {st['ev'].symbol.nunique()} stocks; "
          f"top stocks {st['ev'].symbol.value_counts().head(3).to_dict()}")
    display(compare(st).style.format(FMT))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (s, st) in zip(axes, ins_studies.items()):
    plot_path(ax, st, f"{s}s disclosed, 2016-2021")
fig.tight_layout(); save(fig, "03_insider"); plt.show()

## Findings

Prices are raw closes with ex-date returns dropped. Abnormal return = stock − Nifty. Event studies cover 2016–2023, with 95% intervals from a bootstrap that resamples calendar weeks.

1. **Volume.**
   - 2016–2023 has 50,076 announcements, 26.2% of them tagged important, across 193 categories.
   - Yearly counts grew from 3,858 (2016) to 10,595 (2024), while the important share fell from 34% to 20% (2023).
   - Per stock, the median is 886 (range 21–2,716).
   - The top important categories are Press Release (6,212), Financial Result Updates (1,429), Outcome of Board Meeting (1,153), Updates (958), Acquisition (628) and SEBI takeover disclosures (561).
2. **Event day.** Only 1.5% of announcements can inform the same day's 08:30 forecast. 69.8% inform the next calendar day, and 28.6% wait 2+ calendar days (weekends and holidays). Press releases come mostly during market hours (57.5%); most other important categories come after 15:30 (57–76%).
3. **Events move prices in size, not in a predictable direction.** |CAR[0,+1]| exceeds the random-day control for:

   | Event set | Events | \|CAR[0,+1]\| | Control |
   |---|---|---|---|
   | Financial Result Updates | 1,125 | 3.27% | 1.71% |
   | Outcome of Board Meeting | 872 | 2.94% | 1.67% |
   | Results (financial_results) | 1,125 | 2.36% | 1.67% |
   | All important announcements | 9,977 | 2.11% | 1.70% |
   | Appointment | 285 | 2.08% | 1.72% |
   | Credit Rating | 292 | 1.98% | 1.68% |
   | Change in Director(s) | 314 | 1.89% | 1.51% |
   | Press Release | 4,668 | 1.81% | 1.66% |

   Mean signed CARs at +1, +5 and +10 days are within ±0.6% for every set. Only 2 of 45 directional tests after the event (15 sets × 3 horizons) clear the control: Updates, CAR[0,+5] −0.24%, and Appointment, CAR[0,+10] −0.42%. About 2 would be expected by chance. The share of events with a positive CAR[0,+5] is 44–55% everywhere.
4. **Prices drift up before announcements.** All important announcements show CAR[−5,−1] of +0.21% [+0.10%, +0.33%], and press releases +0.18% [+0.05%, +0.32%], both beyond the control. Companies tend to announce after rises, or the news leaks early.
5. **On day 0, most of the extra move happens after the open.** For results:
   - Mean |abnormal gap| is 0.76% (control 0.41%) and mean |intraday abnormal move| is 1.65% (control 1.13%).
   - The gap is 25% of day-0 variance (13% on random days).
   - A 09:15 entry still sees most of the event-day move, but its direction isn't predictable from the event type.
6. **Results timing.**
   - Release time: 63.5% after 15:30, 35.5% during market hours, 1.1% before 08:30.
   - Results released during market hours already show a raised |AR| on day −1 (1.92% vs 1.19% control), the publication day the time rule can't use.
   - The `financial_results.broadcast_at` time often isn't the first public time. The matching "Financial Result Updates" announcement comes a median 21 hours earlier: on the same day 0 for 45% of results, one trading day earlier for 26%, and 2–10 days earlier for 29%.
   - Where it's a day earlier, the reaction sits on our day −1 (2.79% vs 1.63% on day 0). Where it's 2–10 days earlier, there's no reaction near `broadcast_at` at all.
7. **Event-day volatility.** Mean |daily return| on day 0, against random days:
   - Financial Result Updates 1.97×, Outcome of Board Meeting 1.95×, results 1.52×, Change in Director(s) 1.26×, all important 1.25×, Press Release 1.13×. All of these are beyond the control.
   - Acquisition (1.06×), Credit Rating (1.18×), SEBI takeover disclosures (1.09×) and the smaller categories can't be told apart from random days.
8. **Insider trades and shareholding.**
   - Insider rows fell from 19,236 (2016) to 2,127 (2023), and stocks with data from 43 at most to 26.
   - Promoter market trades fell from 323 to 67 a year. Shareholding exists only from 2021.
   - In 2016–2021, promoter buys (486 events, 20 stocks, 175 of them ADANIPORTS) and sells (387 events, 29 stocks, 179 of them AXISBANK) show no CAR beyond the control at any horizon. For example, after sells AR[0] is −0.14% (difference interval [−0.42%, +0.02%]); after buys CAR[0,+10] is −0.07% [−1.12%, +0.93%].
9. **Sample loss.** 262 of 1,425 results windows (18%) contain a dropped ex-date and are excluded. That biases the sample toward quarters with no dividend in the window, until adjusted prices (task 2.4) exist.

### Which event types move prices beyond the control

- **In size:** results (both sources), board-meeting outcomes, director changes and appointments, credit ratings, press releases, and important announcements overall. Results and board outcomes roughly double the day-0 move.
- **In direction:** none reliably. The two directional hits (Updates and Appointment, both slightly negative) are about what chance would give across 45 tests.
- **Promoter trades:** no effect in 2016–2021, and too little data afterwards.

### What this means
- **Event memory (task 9.1) and the agents.** The category alone says how big a move to expect, not which way it goes. Direction has to come from the content (the results surprise, the text of the announcement), which supports having the agents read the documents.
- **Volatility and stops.** Known upcoming events, especially results dates from board-meeting notices, should widen the volatility forecast and the stop. This feeds the catalyst calendar (7.x).
- **Results time.** Use the earliest of the "Financial Result Updates" and "Outcome of Board Meeting" announcement times as the results time, not `financial_results.broadcast_at`. Added as open item 25 in docs/DATA_COLLECTION.md.
- **Insider features.** Not worth building for the models: the data is thin after 2021 and showed no effect before.